In [18]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!pip install -q transformers evaluate librosa soundfile jiwer

import os
import torch
import pandas as pd
import librosa
import evaluate
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [20]:
BASE_DIR = "/content/drive/MyDrive/JoshTalks_ASR"

FLEURS_AUDIO_DIR = "/content/fleurs_hi_test_audio/test"
TSV_PATH = "/root/.cache/huggingface/hub/datasets--google--fleurs/snapshots/d7c758a6dceecd54a98cac43404d3d576e721f07/data/hi_in/test.tsv"

FINETUNED_MODEL_DIR = os.path.join(BASE_DIR, "04_models", "whisper_small_finetuned_final")

In [21]:
from huggingface_hub import hf_hub_download

test_tsv_path = hf_hub_download(
    repo_id="google/fleurs",
    filename="data/hi_in/test.tsv",
    repo_type="dataset"
)

print("Downloaded TSV path:", test_tsv_path)

Downloaded TSV path: /root/.cache/huggingface/hub/datasets--google--fleurs/snapshots/d7c758a6dceecd54a98cac43404d3d576e721f07/data/hi_in/test.tsv


In [22]:
test_df = pd.read_csv(test_tsv_path, sep="\t", header=None)

test_df.columns = [
    "id", "filename", "raw_transcription", "transcription",
    "char_tokens", "num_samples", "gender"
]

test_df["audio_path"] = test_df["filename"].apply(
    lambda x: os.path.join(FLEURS_AUDIO_DIR, x)
)

sample_df = test_df.head(20).copy()

print("Sample size:", len(sample_df))

Sample size: 20


In [23]:
import os

model_base = "/content/drive/MyDrive/JoshTalks_ASR/04_models/whisper_small_finetuned"

print("All contents:")
print(os.listdir(model_base))

All contents:
['checkpoint-50', 'checkpoint-100', 'checkpoint-150']


In [24]:
import re

checkpoints = [f for f in os.listdir(model_base) if f.startswith("checkpoint")]

# sort by number
checkpoints_sorted = sorted(checkpoints, key=lambda x: int(re.findall(r'\d+', x)[0]))

latest_checkpoint = checkpoints_sorted[-1]

FINETUNED_MODEL_DIR = os.path.join(model_base, latest_checkpoint)

print("Using checkpoint:", latest_checkpoint)
print("Path:", FINETUNED_MODEL_DIR)
print("Files:", os.listdir(FINETUNED_MODEL_DIR))

Using checkpoint: checkpoint-150
Path: /content/drive/MyDrive/JoshTalks_ASR/04_models/whisper_small_finetuned/checkpoint-150
Files: ['config.json', 'generation_config.json', 'model.safetensors', 'training_args.bin', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']


In [25]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the processor from the original base model
processor_ft = WhisperProcessor.from_pretrained("openai/whisper-small")
# Load the fine-tuned model from the checkpoint
model_ft = WhisperForConditionalGeneration.from_pretrained(FINETUNED_MODEL_DIR)

model_ft.to(device)

print("Fine-tuned model loaded.")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Fine-tuned model loaded.


In [26]:
from huggingface_hub import hf_hub_download

audio_tar_path = hf_hub_download(
    repo_id="google/fleurs",
    filename="data/hi_in/audio/test.tar.gz",
    repo_type="dataset"
)

print("Downloaded:", audio_tar_path)

Downloaded: /root/.cache/huggingface/hub/datasets--google--fleurs/snapshots/d7c758a6dceecd54a98cac43404d3d576e721f07/data/hi_in/audio/test.tar.gz


In [27]:
import tarfile
import os

FLEURS_AUDIO_DIR = "/content/fleurs_hi_test_audio"
os.makedirs(FLEURS_AUDIO_DIR, exist_ok=True)

with tarfile.open(audio_tar_path, "r:gz") as tar:
    tar.extractall(FLEURS_AUDIO_DIR)

print("Extracted to:", FLEURS_AUDIO_DIR)

/tmp/ipykernel_9215/3509520709.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(FLEURS_AUDIO_DIR)


Extracted to: /content/fleurs_hi_test_audio


In [28]:
FLEURS_WAV_DIR = os.path.join(FLEURS_AUDIO_DIR, "test")

print("Files sample:", os.listdir(FLEURS_WAV_DIR)[:10])

Files sample: ['2721526147232986835.wav', '15100264608554334458.wav', '3550122948577909974.wav', '17292685997401598277.wav', '4187574626528714176.wav', '11871977786251235384.wav', '5290354952340345069.wav', '5956554166903472780.wav', '9978005649235188652.wav', '12005094579689598791.wav']


In [31]:
test_df["audio_path"] = test_df["filename"].apply(
    lambda x: os.path.join(FLEURS_WAV_DIR, x)
)

#sample_df = test_df.head(20).copy()
sample_df = test_df.head(50).copy()

print("Check exists:", os.path.exists(sample_df.iloc[0]["audio_path"]))

Check exists: True


In [32]:
import librosa
import evaluate
from tqdm import tqdm

wer_metric = evaluate.load("wer")

preds = []
refs = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    audio, _ = librosa.load(row["audio_path"], sr=16000)

    inputs = processor_ft(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    ids = model_ft.generate(inputs)
    pred = processor_ft.batch_decode(ids, skip_special_tokens=True)[0].strip()

    preds.append(pred)
    refs.append(str(row["transcription"]).strip())

ft_wer = wer_metric.compute(predictions=preds, references=refs)

print("Fine-tuned WER:", ft_wer)

100%|██████████| 50/50 [01:44<00:00,  2.09s/it]

Fine-tuned WER: 0.48128807658833767


In [33]:
import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/JoshTalks_ASR"

baseline_csv_path = os.path.join(
    BASE_DIR,
    "05_outputs",
    "baseline_eval",
    "baseline_fleurs_hi_20_predictions.csv"
)

baseline_df = pd.read_csv(baseline_csv_path)

baseline_df.head()

,filename,audio_path,transcription,prediction
0,7251883826251297781.wav,/content/fleurs_hi_test_audio/test/72518838262...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,श्कीं मार्को एक हाएकिं लंबी पैदल याट्रा मार्क ...
1,18194643634199893480.wav,/content/fleurs_hi_test_audio/test/18194643634...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,अगर बादा बादा बादा बादा बादा बादा बादा बादा बा...
2,14585676639063535081.wav,/content/fleurs_hi_test_audio/test/14585676639...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनु विज्यान सहित विज्यान के सबी माबलो पर अर्स्...
3,9520717118868623421.wav,/content/fleurs_hi_test_audio/test/95207171188...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनो विग्यान सहित विग्यान के सभी मामलों पर आरस्...
4,13603726120458761913.wav,/content/fleurs_hi_test_audio/test/13603726120...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मदेयुक के अंथ में पश्छिम यूरोप ने अपनी श्याली ...


In [34]:
results_compare_df = sample_df[["filename", "audio_path", "transcription"]].copy()

results_compare_df["finetuned_pred"] = preds

results_compare_df["is_error"] = results_compare_df["transcription"] != results_compare_df["finetuned_pred"]

results_compare_df.head()

,filename,audio_path,transcription,finetuned_pred,is_error
0,7251883826251297781.wav,/content/fleurs_hi_test_audio/test/72518838262...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स्कीं मार्ग को एक हाइक्किं लंबी पैदल यात्रा मा...,True
1,18194643634199893480.wav,/content/fleurs_hi_test_audio/test/18194643634...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स्किंग मार्को एक हाई किंग मतलब लंबी पैदल याद्र...,True
2,14585676639063535081.wav,/content/fleurs_hi_test_audio/test/14585676639...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनु विग्यान सहित विग्यान के सभी माबलों पर अर्स...,True
3,9520717118868623421.wav,/content/fleurs_hi_test_audio/test/95207171188...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनो विग्यान सहित विग्यान के सभी मामलों पर आरस्...,True
4,13603726120458761913.wav,/content/fleurs_hi_test_audio/test/13603726120...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मत्योग के अंथ में पश्चिम यूरोप ने अपनी शायली व...,True


In [35]:
error_df = results_compare_df[results_compare_df["is_error"] == True].copy()

print("Total errors:", len(error_df))
error_df.head()

Total errors: 50


,filename,audio_path,transcription,finetuned_pred,is_error
0,7251883826251297781.wav,/content/fleurs_hi_test_audio/test/72518838262...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स्कीं मार्ग को एक हाइक्किं लंबी पैदल यात्रा मा...,True
1,18194643634199893480.wav,/content/fleurs_hi_test_audio/test/18194643634...,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स्किंग मार्को एक हाई किंग मतलब लंबी पैदल याद्र...,True
2,14585676639063535081.wav,/content/fleurs_hi_test_audio/test/14585676639...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनु विग्यान सहित विग्यान के सभी माबलों पर अर्स...,True
3,9520717118868623421.wav,/content/fleurs_hi_test_audio/test/95207171188...,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनो विग्यान सहित विग्यान के सभी मामलों पर आरस्...,True
4,13603726120458761913.wav,/content/fleurs_hi_test_audio/test/13603726120...,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मत्योग के अंथ में पश्चिम यूरोप ने अपनी शायली व...,True


In [36]:
N = max(1, len(error_df) // 25)

sampled_errors = error_df.iloc[::N].head(25).copy()

print("Sampled errors:", len(sampled_errors))

Sampled errors: 25


In [37]:
for i, row in sampled_errors.head(10).iterrows():
    print("\nREFERENCE:", row["transcription"])
    print("PREDICTED:", row["finetuned_pred"])
    print("-"*80)


REFERENCE: स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा मार्ग जैसा ही सोचें।
PREDICTED: स्कीं मार्ग को एक हाइक्किं लंबी पैदल यात्रा मार्ग जैसा ही सोचे
--------------------------------------------------------------------------------

REFERENCE: मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्तू के विचारों को स्वीकार किया जाता है
PREDICTED: मनु विग्यान सहित विग्यान के सभी माबलों पर अर्स्तुस्ट के विचारों को सुगार किया जाता है
--------------------------------------------------------------------------------

REFERENCE: मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली विकसित करना शुरू कर दिया धर्मयुद्ध के परिणामस्वरूप उस समय के सबसे बड़े विकास में से एक लोगों ने कपड़ों को बांधने के लिए बटन का उपयोग करना शुरू किया
PREDICTED: मत्योग के अंथ में पश्चिम यूरोप ने अपनी शायली विख्सित करना शरू कर दर्म यूद के पढ़्डनाम सुरूप उस समय के सबसे बढ़े विकास में से एक लोगों ने कपडों का बान्दने कर लिए बटन का अप्योग करना शरू किया
--------------------------------------------------------------------------------

REFER

In [38]:
fixed_preds = []

for _, row in sampled_errors.iterrows():
    audio, _ = librosa.load(row["audio_path"], sr=16000)

    inputs = processor_ft(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)

    ids = model_ft.generate(
        inputs,
        num_beams=5,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        max_length=200
    )

    pred = processor_ft.batch_decode(ids, skip_special_tokens=True)[0].strip()
    fixed_preds.append(pred)

In [39]:
sampled_errors["fixed_pred"] = fixed_preds

comparison_df = sampled_errors[[
    "transcription",
    "finetuned_pred",
    "fixed_pred"
]]

In [40]:
comparison_df.iloc[:8]

,transcription,finetuned_pred,fixed_pred
0,स्कीइंग मार्ग को एक हाईकिंग लंबी पैदल यात्रा म...,स्कीं मार्ग को एक हाइक्किं लंबी पैदल यात्रा मा...,स्कीं मार्ग को एक हाइक्किं लंबी पैदल यात्रा मा...
2,मनोविज्ञान सहित विज्ञान के सभी मामलों पर अरस्त...,मनु विग्यान सहित विग्यान के सभी माबलों पर अर्स...,मनु विग्यान सहित व्यिएन के सभी माबलों पर अर्स्...
4,मध्य युग के अंत में पश्चिमी यूरोप ने अपनी शैली...,मत्योग के अंथ में पश्चिम यूरोप ने अपनी शायली व...,मद्योग के अंथ में पश्चिम यूरोप नहीं अपनी श्याल...
6,यह कहते हुए कि वे चीन के आर्थिक उत्पादन के आधा...,या कहते हुई कि वे चीन के आरते कुटपादन के आदर प...,या कहते हुई कि वे चीन के आर्थे कुत्पादन का आदर...
8,असल में फील्ड ट्रिप साझा करना यात्रा पर विचार ...,अस्सल में फीर ट्रिप साजा करना यात्रा पर विचार ...,अस्सल में फीर्ट्रिप्स आजा करना यात्रा पर विचार...
10,अधिकांश छोटे द्वीप स्वतंत्र राष्ट्र हैं या फ़्...,अधिकाऊन्स छोटेद भी पस्वतंत्र राच्टर हैं या फ्र...,अधिकान्श छोटेद भी पस्वतंत्र राच्टर हैं या फ्रा...
12,पुरुषों को दृढ़ता से नकारें और अपनी बात पर बेह...,पूर्सों को द्रुडता से नकारे और अपनी पात पर पे ...,पूर्सों को द्रुडता से नकारे और अपनी पात पर पे ...
14,इस साल की शुरुआत में 53 साल के कुओमो ने अपना श...,इस थाल की शिरुाथ में तरिप्पन साल के कुवमोंने अ...,इस थाल की शिरुात में ट्रिप्पन सालके कुवमोंने अ...
